# Phase 1 — Harmonisation (features + labels)


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


In [ ]:
import pandas as pd
from driftbench import config, common_core, to_common_core, harmonise_labels, shared_families

feats, labs = {}, {}
for ds in config.DATASETS:
    p = config.INTERIM_DIR / ds
    feats[ds] = pd.read_parquet(p / 'features.parquet')
    labs[ds]  = pd.read_parquet(p / 'labels.parquet')['label']

core = common_core(*feats.values())
print('common-core feature count:', len(core))
print('shared label families:', shared_families(labs[config.DATASETS[0]], labs[config.DATASETS[1]]))

# Persist common-core views + family labels for the cross-dataset demo.
for ds in config.DATASETS:
    out = config.PROCESSED_DIR / ds; out.mkdir(parents=True, exist_ok=True)
    to_common_core(feats[ds], core).to_parquet(out / 'features_core.parquet')
    harmonise_labels(labs[ds]).to_frame('family').to_parquet(out / 'labels_family.parquet')
pd.Series(core).to_csv(config.PROCESSED_DIR / 'common_core_features.csv', index=False)
